# RideMatch Colab Experiment Notebook

This notebook is the research surface for RideMatch. It generates synthetic fulfillment data, inspects the simulation, establishes the nearest-driver baseline, and provides a place to test improved ranking objectives.

The current `matched` label is generated by the nearest-driver policy. A model trained on it validates the pipeline but mostly learns to imitate that policy. Do not claim an ML improvement until policies are compared at the request level using wait, SLA, coverage, cancellation, and utilization metrics.

In [ ]:
# Colab setup. Skip the clone cell when running locally from the repository.
# !git clone https://github.com/sohan2000/ride-match-ml.git
# %cd ride-match-ml
# !pip install -r requirements.txt

from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT.resolve()}")

In [ ]:
import pandas as pd

from src.simulator.generate_data import generate_dataset

candidates = generate_dataset(
    num_drivers=100,
    num_riders=10_000,
    steps=1_440,
    seed=42,
)

candidates.head()

In [ ]:
print("Rows:", len(candidates))
print("Requests:", candidates["request_id"].nunique())
print("Candidate positive rate:", candidates["matched"].mean())
print("Candidates per request:")
print(candidates.groupby("request_id").size().describe())

display(candidates[[
    "distance_km",
    "eta_minutes",
    "driver_idle_minutes",
    "available_drivers",
    "open_requests",
    "hour_of_day",
]].describe())

In [ ]:
from src.utils.metrics import marketplace_metrics
from src.simulator.generate_data import generate_outcomes

outcomes = generate_outcomes(
    num_drivers=100,
    num_riders=10_000,
    steps=1_440,
    seed=42,
)

baseline_metrics = marketplace_metrics(outcomes.to_dict("records"), sla_minutes=5.0)
pd.Series(baseline_metrics, name="nearest_driver_baseline")

## Improved objective to implement next

The baseline label is policy-generated. For a meaningful ML experiment, add a delayed candidate utility or acceptance outcome that is not simply defined as nearest driver. Candidate selection should then be evaluated by replaying the policy over the same requests and comparing request-level KPIs.